**Prérequis impératif :** le notebook 01 doit produire un texte épuré approuvé et son manifeste. Le notebook 02 refuse d'utiliser directement le corpus brut pour mesurer les métriques. Conserver les deux empreintes et les décisions dans les résultats.


# PT-LLMvsPy · Notebook 02 — Mettre à l'épreuve les métriques globales de Gemini

**Question de recherche.** Les indicateurs quantitatifs annoncés par Gemini sur *Le Féminisme* d'Émile Faguet sont-ils reproductibles lorsque la définition des mots, des formes et des phrases est rendue explicite ?

**Ce notebook ne donne pas de corrigé des nombres de Faguet.** Vous allez vérifier les fonctions communes sur **un extrait que vous aurez choisi**, enregistrer les réussites et anomalies, proposer les corrections nécessaires par pull request, puis seulement appliquer le protocole au texte entier. Une correspondance numérique ne prouve pas que Gemini a effectivement utilisé ce protocole.

**🟦 Cadre fourni par l'enseignant** : corpus et affirmations figés, questions de recherche, ressources.
**🟪 À comprendre** : notions et hypothèses.
**🟩 Votre travail** : choisir, prédire, tester, corriger, argumenter.
**🟧 Vigilance** : résultats non comparables, erreurs, limites.
**⬛ Trace attendue** : tableau, extrait, rapport ou commit.

**Périmètre du notebook 02** : huit valeurs affichées pour six indicateurs globaux (tokens, types, TTR avec son %, hapax avec son %, phrases, longueur moyenne), ainsi que les commentaires interprétatifs accompagnant le tableau. **Les intervalles G1–G6 de cooccurrence proviennent d'une autre transcription et ne sont pas l'objet de ce notebook.**

## 0 · Contrat de travail et installation

🟦 **Architecture** — les fonctions sont importées depuis `tal_toolkit` ; les étudiants ne doivent pas maintenir une copie divergente de leur code dans ce cahier.

Ce notebook se lance de préférence **depuis un clone du dépôt PT-LLMvsPy**, sur la branche de la PR en cours. Dans le répertoire du dépôt : `python -m pip install -e ".[test,nlp]"`, puis ouvrir `notebooks/02_mesures_lexicales.ipynb` avec Jupyter. Le dépôt est privé : un notebook isolé ouvert dans Colab ne pourra pas l'installer par magie, ni sans accès GitHub. Aucun identifiant ni jeton ne doit être écrit dans le cahier.

Le protocole de départ est une **proposition d'expérience**, pas celui de Gemini : spaCy français sans modèle statistique, segmentation `sentencizer`, tokens alphabétiques, formes en minuscules et retours de ligne conservés. Toute variante doit être étiquetée séparément.

In [ ]:
from pathlib import Path
import sys
import json
from pprint import pprint

# Retrouver un dépôt local lorsque Jupyter démarre dans notebooks/ ou à sa racine.
racine = next((p for p in [Path.cwd(), Path.cwd().parent]
               if (p / "src" / "tal_toolkit" / "__init__.py").exists()), None)
if racine is None:
    raise RuntimeError("Ouvrir ce cahier dans un clone PT-LLMvsPy puis installer la bibliothèque.")
sys.path.insert(0, str(racine / "src"))
from tal_toolkit import (
    charger_corpus, ProtocoleGlobal, preparer_texte_pour_analyse,
    calculer_metriques_globales, comparer_annonces
)
import spacy
print("Dépôt :", racine)
print("Version spaCy :", spacy.__version__)

## 1 · Définir ce que l'on cherche à vérifier

🟦 **Origine de l'affirmation** : la section « 1. Cadre Méthodologique et Métriques Globales du Corpus » a été transmise séparément par l'enseignant le 10 novembre 2026. Elle ne figure pas dans la transcription des trois prompts précédemment utilisée dans les TD S3.

| Indicateur annoncé | Valeur Gemini |
|---|---:|
| Total de tokens | 79 540 |
| Types | 8 633 |
| TTR | 0,1085 (10,85 %) |
| Hapax | 4 609 (53,39 % des types) |
| Phrases | 3 829 |
| Moyenne tokens / phrase | 20,77 |

🟪 **Termes à maîtriser** : un token est une unité issue d'une tokenisation (pas nécessairement un mot linguistique), un type est une forme distincte dans la convention choisie, un hapax est un **type** observé une seule fois, le TTR vaut `types/tokens`, et la longueur moyenne déclarée vaut `tokens/phrases`. Il faut annoncer si ponctuation, nombres, apostrophes, tirets et casse sont conservés. La **segmentation en phrases** est également une décision méthodologique.

🟧 **Attention à la dépendance des valeurs** : les trois ratios sont calculables à partir des quatre effectifs. Leur cohérence interne ne démontre donc pas la réalité de l'analyse du texte.

In [ ]:
annonces_path = racine / "data" / "annonces_globales_gemini.json"
annonces = json.loads(annonces_path.read_text(encoding="utf-8"))
assert annonces["version"] == "gemini_metrics_globales_v1"
print("Source des chiffres :", annonces["origine"])
for ligne in annonces["annonces"]:
    print(f'{ligne["libelle"]:<30} {ligne["valeur"]:<12} {ligne["unite"]}')

### 1.1 — Premier contrôle : cohérence **arithmétique** de la réponse Gemini

🟩 **Votre tâche** : avant l'exécution, prévoir les valeurs des trois formules à l'aide des effectifs annoncés. Puis exécuter le calcul et commenter ce qu'il permet — et **ne permet pas** — de conclure.

⬛ **Trace** : expliquer pourquoi l'accord entre TTR annoncé et TTR recalculé à partir des chiffres annoncés ne valide pas le décompte du livre.

In [ ]:
chiffres_gemini = {a["cle"]: a["valeur"] for a in annonces["annonces"]}
n = chiffres_gemini["tokens"]
v = chiffres_gemini["types"]
h = chiffres_gemini["hapax"]
s = chiffres_gemini["phrases"]
recalcul_arithmetique = {
    "metriques": {
        **{k: chiffres_gemini[k] for k in ["tokens", "types", "hapax", "phrases"]},
        "ttr": v / n,
        "ttr_pourcent": 100 * v / n,
        "hapax_types_pourcent": 100 * h / v,
        "mots_par_phrase": n / s,
    }
}
pprint(comparer_annonces(recalcul_arithmetique, annonces["annonces"]))
print("Ce contrôle n'a pas ouvert le corpus Faguet.")

## 2 · Récupérer le **même** texte que dans les TD S3

🟦 **Source figée** : `faguet_source.txt` est la version nettoyée utilisée dans les TD S3. Son historique de nettoyage antérieur n'est pas intégralement connu. Le fichier est téléchargé depuis un **commit fixé**, puis contrôlé par SHA-256 ; les fins de ligne sont préservées.

🟧 L'édition imprimée et ce fichier ne sont pas interchangeables. La comparaison avec Gemini ne sera pleinement possible que si le fichier fourni à Gemini était identique. La réponse citée ne documente pas ce point avec une empreinte vérifiable.

In [ ]:
from urllib.request import urlopen
from urllib.parse import quote
import hashlib

CHEMIN = "Notebooks TD/S3/ressources/faguet_source.txt"
COMMIT_COURS = "b92e92b6c216014283056c83aa545881a447b3a5"
SHA256_ATTENDU = "c9832aecd99b11f3c769d11173fd6039a4948a8ee86484658dd8c7a25848a8fa"
adresse = ("https://raw.githubusercontent.com/dreymond732/tal-notebook-evaluator/"
           + COMMIT_COURS + "/" + quote(CHEMIN))
destination = racine / "ressources_locales" / "faguet_source.txt"
destination.parent.mkdir(exist_ok=True)
if not destination.exists():
    with urlopen(adresse, timeout=45) as reponse:
        destination.write_bytes(reponse.read())
empreinte_verifiee = hashlib.sha256(destination.read_bytes()).hexdigest()
assert empreinte_verifiee == SHA256_ATTENDU, (
    "Échec d'empreinte : fichier différent ; ne pas écraser sans diagnostic."
)
source_originale = charger_corpus(destination, sha256_attendu=SHA256_ATTENDU)

# Point de jonction obligatoire avec le notebook 01.
dossier_epuration = racine / "resultats_epuration_faguet"
manifeste_epuration = dossier_epuration / "faguet_epure.manifest.json"
fichier_epure = dossier_epuration / "faguet_epure.txt"
assert manifeste_epuration.exists() and fichier_epure.exists(), (
    "Exécutez et approuvez d'abord le notebook 01 ; placez les fichiers "
    "exportés dans resultats_epuration_faguet/."
)
manifeste = json.loads(manifeste_epuration.read_text(encoding="utf-8"))
assert manifeste["source"]["sha256_original"] == SHA256_ATTENDU
assert all(etat == "approuve"
           for etat in manifeste["validation"]["decisions"].values())
corpus = charger_corpus(
    fichier_epure,
    sha256_attendu=manifeste["export"]["sha256_texte_utf8"],
)
print("Empreinte source :", source_originale.sha256)
print("Empreinte épurée :", corpus.sha256)
print("Caractères dans le corpus approuvé :", len(corpus.texte))


## 3 · Choisir un échantillon et écrire ses prédictions avant calcul

🟩 **Votre tâche** : sélectionner **vous-même** dans Faguet un court passage suffisamment riche : répétitions, mots commençant par une majuscule, ponctuation, apostrophes et, si possible, un retour de ligne typographique. Vous devrez pouvoir compter ses mots et ses formes **à la main**. Conserver les bornes de caractères permet de le retrouver sans ambiguïté.

Pour vous orienter, visualisez quelques passages, puis remplacez les valeurs `None` des deux bornes. La sélection est votre décision ; aucune tranche n'est imposée par ce cahier.

⬛ **Trace** : pourquoi avoir choisi ce passage ? Quelles difficultés prévoyez-vous pour la tokenisation et les phrases ?

In [ ]:
for debut_candidat in [0, len(corpus.texte) // 4, len(corpus.texte) // 2]:
    print("\n--- Position", debut_candidat, "---")
    print(repr(corpus.texte[debut_candidat:debut_candidat + 280]))
# Votre choix, à documenter :
DEBUT_EXTRAIT = None
FIN_EXTRAIT = None
JUSTIFICATION_EXTRAIT = ""

In [ ]:
assert isinstance(DEBUT_EXTRAIT, int) and isinstance(FIN_EXTRAIT, int), (
    "Vous devez choisir les bornes DEBUT_EXTRAIT et FIN_EXTRAIT."
)
assert 0 <= DEBUT_EXTRAIT < FIN_EXTRAIT <= len(corpus.texte)
assert 80 <= FIN_EXTRAIT - DEBUT_EXTRAIT <= 900, (
    "Choisir un court extrait contrôlable manuellement (80 à 900 caractères)."
)
assert JUSTIFICATION_EXTRAIT.strip(), "Justifier le choix de l'extrait."
extrait = corpus.texte[DEBUT_EXTRAIT:FIN_EXTRAIT]
print("Extrait et bornes :", DEBUT_EXTRAIT, FIN_EXTRAIT)
print(repr(extrait))

### 3.1 — Conventions du premier essai

🟪 **Notion** : ce sont des décisions sur la méthode, **non** une découverte du protocole Gemini.

- `alphabetiques` : seuls les tokens dont `is_alpha` est vrai sont des mots comptés ;
- `minuscules` : `Chat` et `chat` représentent un même type ;
- `conserver` : le texte, notamment ses CRLF, n'est pas retouché ;
- `sentencizer` : séparation en phrases par une règle spaCy (sans analyse syntaxique statistique) ;
- on ignore dans le compte des phrases les segments sans token retenu.

🟧 L'apostrophe et le trait d'union peuvent modifier le nombre de tokens ; les retours à la ligne peuvent être confondus avec des frontières de phrases. Les ambiguïtés observées doivent figurer dans votre diagnostic.

In [ ]:
protocole = ProtocoleGlobal(
    tokens="alphabetiques",
    formes="minuscules",
    retours="conserver",
    ignorer_phrases_sans_mots=True,
)
nlp = spacy.blank("fr")
nlp.add_pipe("sentencizer")
nlp.max_length = max(nlp.max_length, len(corpus.texte) + 1)
print("Protocole :", protocole)
print("Pipeline :", nlp.pipe_names)

### 3.2 — Écrire les quatre effectifs manuels

🟩 **Votre tâche** : sur l'extrait **dans la convention annoncée**, prédire et documenter séparément `tokens`, `types`, `hapax` et `phrases`. Établir une liste de formes ou une petite table de fréquences à l'appui. Un « mot » séparé par un espace n'est pas nécessairement un token spaCy.

⬛ **Trace** : enregistrer les effectifs prédits **avant** d'appeler la fonction de mesure, ainsi que les cas discutables (apostrophes, abréviations, titres, ponctuation).

In [ ]:
# À renseigner MANUELLEMENT et AVANT l'exécution de la cellule de calcul.
ATTENDU_MANUEL = {
    "tokens": None,
    "types": None,
    "hapax": None,
    "phrases": None,
}
NOTES_COMPTAGE_MANUEL = ""

## 4 · Tester les fonctions de la bibliothèque sur votre échantillon

🟩 **Votre tâche** : contrôler **successivement** la préparation du texte, les quatre effectifs, les ratios calculés et les positions des éléments affichés. Les valeurs manuelles servent d'oracle **local**, pas les chiffres de Gemini.

⬛ **Trace** : pour chaque écart, enregistrer *attendu, observé, localisation, diagnostic, nature de l'écart* (bug ou convention discutable).

In [ ]:
# Fonction 1 : préparation, sans perte de caractères ni décalage d'offset.
texte_prepare = preparer_texte_pour_analyse(extrait, protocole.retours)
assert len(texte_prepare) == len(extrait)
print("Retours conservés :", texte_prepare == extrait)

# Fonction 2 : métriques globales avec traces reproductibles.
mesure_extrait = calculer_metriques_globales(
    extrait, nlp, protocole=protocole, max_exemples=1000
)
assert all(mesure_extrait["controles"].values())
pprint(mesure_extrait["metriques"])
print("Tokens retenus — formes et positions :")
for item in mesure_extrait["traces"]["tokens"]:
    print(item["debut"], item["fin"], repr(item["forme_originale"]),
          "=>", repr(item["forme_comptee"]))
print("Phrases observées — positions, nombre de mots :")
for item in mesure_extrait["traces"]["phrases"]:
    print(item["debut"], item["fin"], item["tokens_retenus"],
          repr(item["extrait_original"][:200]))

In [ ]:
assert NOTES_COMPTAGE_MANUEL.strip(), "Documenter votre méthode de comptage manuel."
assert all(type(v) is int and v >= 0 for v in ATTENDU_MANUEL.values()), (
    "Renseigner d'abord les quatre effectifs ATTENDU_MANUEL avec des entiers."
)
# Fonction 3 : comparer les quatre comptes et les trois proportions DÉRIVÉES.
t, v, h, s = (
    ATTENDU_MANUEL[k] for k in ["tokens", "types", "hapax", "phrases"]
)
attendus_derivees = {
    "ttr": None if not t else v / t,
    "ttr_pourcent": None if not t else 100 * v / t,
    "hapax_types_pourcent": None if not v else 100 * h / v,
    "mots_par_phrase": None if not s else t / s,
}
assert 0 <= h <= v <= t, "Vos effectifs manuels sont incohérents : à revoir."
annonces_manuelles = [
    {"cle": k, "valeur": val, "decimales": 4 if k == "ttr" else
        2 if k in attendus_derivees else 0, "unite": "échantillon manuel"}
    for k, val in {**ATTENDU_MANUEL, **attendus_derivees}.items()
    if val is not None
]
comparaison_extrait = comparer_annonces(
    mesure_extrait, annonces_manuelles, origine_reference="comptage_manuel"
)
for ligne in comparaison_extrait:
    print(ligne["cle"], "manuel =", ligne["valeur_reference"],
          "| calcul =", ligne["observe_affiche"],
          "|", ligne["statut_numerique"])
ecarts = [l for l in comparaison_extrait if l["statut_numerique"] != "MEME_VALEUR_A_L_ARRONDI"]
print("Nombre d'écarts à analyser :", len(ecarts))

### 4.1 — Diagnostiquer sans maquiller le résultat

🟧 **Un désaccord ne signifie pas automatiquement que la fonction est boguée** : vérifiez d'abord que le contrôle manuel applique bien le même protocole. Distinguez :

1. **Erreur de code** : le programme contredit sa définition annoncée ;
2. **Ambiguïté de convention** : espaces, abréviations, mots composés, majuscules, variantes de segmentation ;
3. **Erreur du contrôle manuel** : l'oracle initial doit lui aussi pouvoir être corrigé avec justification.

🟩 Vérifiez spécialement les apostrophes, les mots avec tiret, les titres et la frontière des phrases. Si nécessaire, ouvrez une **PR de correction** dans la bibliothèque, en ajoutant le test qui montre le défaut. Gardez les observations précédant la correction et celles obtenues après.

⬛ **Trace de revue** : tableau des anomalies et décision argumentée (corriger / conserver / ouvrir une question).

In [ ]:
# À renseigner : au moins un exemple observable, même si aucun bug n'est trouvé.
JOURNAL_ANOMALIES = [
    # {"element": "...", "attendu": "...", "observe": "...",
    #  "preuve": "bornes ou extrait", "diagnostic": "bug|convention|oracle",
    #  "decision": "...", "pr": None},
]
TESTS_DOCUMENTES = ""
VALIDATION_ECHANTILLON = False  # True seulement après validation de vos tests et traces.

## 5 · Analyser le texte intégral **après la validation de l'échantillon**

🟩 Lancer les fonctions communes sur le même fichier figé avec le protocole éprouvé. Conserver le **SHA-256**, les versions logicielles, les unités et les traces. Ne jamais modifier les valeurs annoncées par Gemini pour faire correspondre les résultats.

🟧 **Limite de comparabilité** : ne pas déclarer « Gemini s'est trompé » tant que son protocole (et l'identité exacte du fichier utilisé) n'est pas documenté.

In [ ]:
assert VALIDATION_ECHANTILLON is True, (
    "Valider votre échantillon et documenter les anomalies avant le corpus complet."
)
mesure_corpus = calculer_metriques_globales(
    corpus, nlp, protocole=protocole, max_exemples=12
)
assert all(mesure_corpus["controles"].values())
print("Provenance :", mesure_corpus["provenance"])
print("Version et pipeline :", mesure_corpus["logiciel"])
print("Conventions :", mesure_corpus["protocole"])
print("Métriques mesurées :")
pprint(mesure_corpus["metriques"])

## 6 · Confronter **tous les chiffres du tableau** à vos résultats

🟩 Pour chaque ligne : retenir la valeur brute, la valeur affichée à la précision indiquée par Gemini, l'unité et le statut d'écart numérique. **Conserver les résultats qui ne correspondent pas** ; ne pas adapter les conventions dans cette cellule.

⬛ **Trace** : tableau de confrontation de huit valeurs affichées (six indicateurs, dont TTR et hapax ont une seconde présentation en pourcentage), assorti d'un commentaire sur la comparabilité.

In [ ]:
confrontation = comparer_annonces(mesure_corpus, annonces["annonces"])
print(f'{"Indicateur":<29} {"Gemini":>14} {"Observé":>17}  Statut numérique')
for ligne in confrontation:
    print(f'{ligne["cle"]:<29} {str(ligne["valeur_reference"]):>14} '
          f'{str(ligne["observe_affiche"]):>17}  {ligne["statut_numerique"]}')
print("\nProtocole Gemini : INCONNU (pour toutes les lignes).")

## 7 · Tester la sensibilité aux définitions

🟪 **Question méthodologique** : un changement de tokenisation ou de segmentation change-t-il les conclusions ? Ce contrôle ne cherche pas le réglage qui « colle » le mieux aux nombres de Gemini.

🟩 **Votre tâche** : avant de lancer les variantes, formuler ce que vous attendez de chacune : tokens non ponctuation, distinction de casse, recomposition des retours typographiques. Chaque ligne est une expérience différente ; consigner ses paramètres.

⬛ **Trace** : comparer les quatre effectifs fondamentaux des variantes, puis interpréter les écarts. Une comparaison équitable du TTR entre extraits de longueurs différentes n'est pas assurée (McCarthy & Jarvis, 2010).

In [ ]:
variantes = [
    ("baseline", protocole),
    ("tous_les_non_ponctuation", ProtocoleGlobal(
        tokens="sans_ponctuation", formes="minuscules", retours="conserver")),
    ("casse_distinguee", ProtocoleGlobal(
        tokens="alphabetiques", formes="exactes", retours="conserver")),
    ("lignes_recomposees", ProtocoleGlobal(
        tokens="alphabetiques", formes="minuscules", retours="recomposer_lignes")),
]
sensibilite = []
for nom, variante in variantes:
    resultat = calculer_metriques_globales(
        corpus, nlp, protocole=variante, max_exemples=0
    )
    sensibilite.append({
        "scenario": nom, "protocole": resultat["protocole"],
        "metriques": resultat["metriques"],
    })
for ligne in sensibilite:
    m = ligne["metriques"]
    print(f'{ligne["scenario"]:<25} tokens={m["tokens"]:<8} '
          f'types={m["types"]:<8} hapax={m["hapax"]:<8} '
          f'phrases={m["phrases"]:<8}')

## 8 · Interpréter sans extrapoler

🟪 **À comprendre** : le TTR est sensible à la taille du texte ; un TTR faible ne prouve pas, à lui seul, une « figeabilité thématique ». Une moyenne de mots par phrase ne mesure pas directement la complexité syntaxique ; un taux d'hapax ne caractérise pas automatiquement la diversité des exemples.

🟩 **Votre travail** : relire les cinq conclusions de Gemini enregistrées dans `conclusions_a_examiner`. Pour chacune, distinguer *ce que nos métriques permettent d'observer*, *l'interprétation proposée par Gemini* et *ce qui exigerait d'autres analyses*.

⬛ **Trace** : un tableau « affirmation / données disponibles / appréciation / preuve / limites » et un paragraphe explicitant la prudence interprétative.

**Références vérifiables**

- McCarthy, P. M., & Jarvis, S. (2010). MTLD, vocd-D, and HD-D: A validation study of sophisticated approaches to lexical diversity assessment. *Behavior Research Methods, 42*(2), 381–392. https://doi.org/10.3758/BRM.42.2.381
- Koizumi, R., & In'nami, Y. (2012). Effects of text length on lexical diversity measures: Using short texts with less than 200 tokens. *System, 40*(4), 554–564. https://doi.org/10.1016/j.system.2012.10.012
- Explosion. (s. d.). *Token* (spaCy API). https://spacy.io/api/token
- Explosion. (s. d.). *Sentencizer* (spaCy API). https://spacy.io/api/sentencizer

In [ ]:
# Compléter avec vos formulations et preuves avant export.
TABLEAU_INTERPRETATION = [
    # {"affirmation": "...", "observe": "...", "preuve": "...",
    #  "interpretation": "...", "limite": "..."},
]
CONCLUSION_CRITIQUE = ""
QUESTION_REPRISE_SEMAINE_SUIVANTE = ""

## 9 · Export de preuves et revue hebdomadaire

🟩 Préparer une restitution courte : choix de l'échantillon, résultat de vos essais, anomalies rencontrées, corrections proposées, valeurs sur corpus entier, confrontation des nombres Gemini et questions encore ouvertes.

⬛ **Livrables** : notebook avec essais et commentaires, fichier JSON de résultats, PR éventuelle documentée, compte rendu de revue.

🟦 **Point de liaison avec le notebook 02** : la revue hebdomadaire décide quelles analyses lexicales (mots dominants, formes, lemmes, couverture du vocabulaire) seront approfondies et quelles méthodes devront être contrôlées plus finement. Les cooccurrences par phrase ou fenêtre relèvent d'une phase ultérieure.

🟧 Ne pas exporter de jeton GitHub, identifiant personnel ou clé d'accès.

In [ ]:
assert CONCLUSION_CRITIQUE.strip(), "Rédiger votre conclusion critique."
assert QUESTION_REPRISE_SEMAINE_SUIVANTE.strip(), "Formuler une question pour la revue."
sortie = {
    "projet": "PT-LLMvsPy",
    "notebook": "02_mesures_lexicales",
    "tableau_llm": annonces["version"],
    "corpus": mesure_corpus["provenance"],
    "logiciel": mesure_corpus["logiciel"],
    "protocole": mesure_corpus["protocole"],
    "extrait": {
        "debut": DEBUT_EXTRAIT, "fin": FIN_EXTRAIT,
        "justification": JUSTIFICATION_EXTRAIT,
        "attendu_manuel": ATTENDU_MANUEL,
        "mesure": mesure_extrait,
        "ecarts": ecarts,
        "journal_anomalies": JOURNAL_ANOMALIES,
        "tests_documentes": TESTS_DOCUMENTES,
    },
    "mesures_corpus": mesure_corpus,
    "confrontation_gemini": confrontation,
    "sensibilite": sensibilite,
    "interpretation": TABLEAU_INTERPRETATION,
    "conclusion_critique": CONCLUSION_CRITIQUE,
    "suite": QUESTION_REPRISE_SEMAINE_SUIVANTE,
}
fichier_sortie = racine / "resultats_notebook02.json"
fichier_sortie.write_text(
    json.dumps(sortie, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Export enregistré :", fichier_sortie)